###### JSON 형식 출력 파서



In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser-JSON")

model = ChatOpenAI(model="gpt-4o-mini",temperature=0)

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser-JSON


In [2]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [ ]:
question = "지구 온난화의 심각성에 대해 알려주세요."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [4]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
chain = prompt | model | parser
answer = chain.invoke({"question": question})


In [5]:
answer["description"]

'지구 온난화는 지구의 평균 기온이 상승하는 현상으로, 기후 변화, 해수면 상승, 생태계 파괴 등의 심각한 영향을 미칩니다.'

In [6]:
answer

{'description': '지구 온난화는 지구의 평균 기온이 상승하는 현상으로, 기후 변화, 해수면 상승, 생태계 파괴 등의 심각한 영향을 미칩니다.',
 'hashtags': '#지구온난화 #기후변화'}